# Spark Stream, DStreams

## Local Spark setup


In [ ]:
# Local Spark setup for the book examples.
from pathlib import Path
import shutil
from itertools import count
from pyspark.streaming import StreamingContext
from pyspark.sql import SparkSession

DATA_DIR = Path.cwd()
OUTPUT_DIR = DATA_DIR / "_spark_output" / "dstreams"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("spark-intro-dstreams")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "false")
)
spark = builder.getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
sqlContext = spark

_stream_counter = count()

def get_streaming_context(batch_duration=1):
    checkpoint_dir = OUTPUT_DIR / f"checkpoint-{next(_stream_counter)}"
    shutil.rmtree(checkpoint_dir, ignore_errors=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    ssc = StreamingContext(sc, batch_duration)
    ssc.checkpoint(str(checkpoint_dir))
    return ssc


## Transformations

### Map

In [ ]:
from pyspark.streaming import StreamingContext
from random import choice

ssc = get_streaming_context()

alphabets = list('abcdefghijklmnopqrstuvwxyz')
input_data = [[choice(alphabets) for _ in range(100)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).map(lambda word: (word, 1))
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Flat map

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[i for i in range(100)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).flatMap(lambda num: [(num, randint(1, 10)) for _ in range(num)])
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Filter

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[randint(1, 100) for i in range(100)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).filter(lambda num: num % 2 == 0)
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Repartition

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[randint(1, 100) for i in range(100)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).repartition(1)
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Union

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data1 = [[randint(1, 100) for i in range(100)] for _ in range(100)]
input_data2 = [[randint(1, 100) for i in range(100)] for _ in range(100)]

rdd_queue1 = [ssc.sparkContext.parallelize(item) for item in input_data1]
rdd_queue2 = [ssc.sparkContext.parallelize(item) for item in input_data2]

stream1 = ssc.queueStream(rdd_queue1).filter(lambda num: num % 2 == 0)
stream2 = ssc.queueStream(rdd_queue2).filter(lambda num: num % 2 == 0)

stream = stream1.union(stream2)
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Count

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[randint(1, 100) for _ in range(randint(1, 20))] for i in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).count()
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Reduce

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[randint(1, 100) for _ in range(randint(1, 20))] for i in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).reduce(lambda a, b: a + b)
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Count by value

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[randint(1, 100) for _ in range(randint(1, 20))] for i in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).countByValue()
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Reduce by key

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[(randint(1, 2), randint(1, 100)) for _ in range(randint(1, 20))] for i in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).reduceByKey(lambda a, b: a + b)
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Join

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint, choice

ssc = get_streaming_context()

input_data1 = [[(choice([0, 1]), randint(1, 2)) for _ in range(5)] for _ in range(100)]
input_data2 = [[(choice([0, 1]), choice(['a', 'b'])) for _ in range(5)] for _ in range(100)]

rdd_queue1 = [ssc.sparkContext.parallelize(item) for item in input_data1]
rdd_queue2 = [ssc.sparkContext.parallelize(item) for item in input_data2]

counts1 = ssc.queueStream(rdd_queue1)
counts2 = ssc.queueStream(rdd_queue2)

stream = counts1.join(counts2)
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Cogroup

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint, choice

ssc = get_streaming_context()

input_data1 = [[(choice([0, 1]), randint(1, 2)) for _ in range(5)] for _ in range(100)]
input_data2 = [[(choice([0, 1]), choice(['a', 'b'])) for _ in range(5)] for _ in range(100)]

rdd_queue1 = [ssc.sparkContext.parallelize(item) for item in input_data1]
rdd_queue2 = [ssc.sparkContext.parallelize(item) for item in input_data2]

counts1 = ssc.queueStream(rdd_queue1)
counts2 = ssc.queueStream(rdd_queue2)

stream = counts1.cogroup(counts2)
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Transform

In [ ]:
from pyspark.streaming import StreamingContext
from random import randint

ssc = get_streaming_context()

input_data = [[i for i in range(100)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]

stream = ssc.queueStream(rdd_queue).transform(lambda rdd: rdd.filter(lambda x: x % 2 == 0))
stream.pprint()

ssc.start()
ssc.stop(stopSparkContext=False, stopGraceFully=True)


## Window operations

### Window

In [ ]:
from pyspark.streaming import StreamingContext
from time import sleep

ssc = get_streaming_context()

input_data = [[c for c in 'abcde'] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]
words = ssc.queueStream(rdd_queue)
pairs = words.map(lambda word: (word, 1))

stream = pairs.window(5, 1)
stream.pprint()

ssc.start()
sleep(3)
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Count by window

In [ ]:
from pyspark.streaming import StreamingContext
from time import sleep

ssc = get_streaming_context()

input_data = [[c for c in 'abcde'] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]
words = ssc.queueStream(rdd_queue)
pairs = words.map(lambda word: (word, 1))

stream = pairs.countByWindow(5, 1)
stream.pprint()

ssc.start()
sleep(3)
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Reduce by window

In [ ]:
from pyspark.streaming import StreamingContext
from time import sleep

ssc = get_streaming_context()

input_data = [[c for c in 'abcde'] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]
words = ssc.queueStream(rdd_queue)
pairs = words.map(lambda word: (word, 1))

stream = pairs.reduceByWindow(lambda a, b: a + b, lambda a, b: a - b, 5, 1)
stream.pprint()

ssc.start()
sleep(3)
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Reduce by key and window

In [ ]:
from pyspark.streaming import StreamingContext
from time import sleep

ssc = get_streaming_context()

input_data = [[str(i) for i in range(10)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]
words = ssc.queueStream(rdd_queue)
pairs = words.map(lambda word: (word, 1))

stream = pairs.reduceByKeyAndWindow(lambda a, b: a + b, lambda a, b: a - b, 5, 1)
stream.pprint()

ssc.start()
sleep(3)
ssc.stop(stopSparkContext=False, stopGraceFully=True)


### Count by value and window

In [ ]:
from pyspark.streaming import StreamingContext
from time import sleep

ssc = get_streaming_context()

input_data = [[str(i) for i in range(10)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]
words = ssc.queueStream(rdd_queue)
pairs = words.map(lambda word: (word, 1))

stream = pairs.countByValueAndWindow(5, 1)
stream.pprint()

ssc.start()
sleep(3)
ssc.stop(stopSparkContext=False, stopGraceFully=True)


## Output operations

### Save as text file

In [ ]:
from pyspark.streaming import StreamingContext
from time import sleep

ssc = get_streaming_context()

input_data = [[str(i) for i in range(10)] for _ in range(100)]
rdd_queue = [ssc.sparkContext.parallelize(item) for item in input_data]
words = ssc.queueStream(rdd_queue)
pairs = words.map(lambda word: (word, 1))

stream = pairs.countByValueAndWindow(5, 1)
stream.pprint()
stream.saveAsTextFiles(str(OUTPUT_DIR / 'dstream' / 'data'), 'txt')

ssc.start()
sleep(3)
ssc.stop(stopSparkContext=False, stopGraceFully=True)
